# Assignment 6: Build and Evaluate Tree Models

Juan Maldonado Franco  
DDS-8555 Predictive Analysis  
Mohamed Nabeel

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd()
for parent in [ROOT, *ROOT.parents]:
    if (parent / "DDS-8555 - Predictive Analysis").exists():
        COURSE = parent / "DDS-8555 - Predictive Analysis"
        break
else:
    COURSE = ROOT.parents[1]
DATA = COURSE / "data"
KAGGLE = DATA / "kaggle"
SUBMISSIONS = DATA / "submissions"
RANDOM_STATE = 42
pd.set_option("display.max_columns", 80)

## Conceptual Question 1

A six-region recursive binary split can be created by first splitting X1 at t1, then splitting the left side on X2 at t2, and continuing with additional splits inside selected regions.  The matching decision tree starts with the first X1 split at the root, then branches into the later X2 and X1 cuts.  The important point is that each rectangular region corresponds to one terminal node, and each internal node corresponds to one binary decision.

## Applied Question 12: Tree Methods and BART

The textbook applied exercise asks for boosting, bagging, random forests, and BART on a chosen data set.  The built-in diabetes regression data fits this part because BART is naturally implemented here as a regression model through `ISLP.bart.BART`.  RMSE and R-squared compare the flexible tree methods on the same held-out set.

In [2]:
from ISLP.bart import BART
from sklearn.datasets import load_diabetes
from sklearn.ensemble import BaggingRegressor, GradientBoostingRegressor, RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor

diabetes = load_diabetes(as_frame=True)
X_diabetes = diabetes.data
y_diabetes = diabetes.target
X_train_d, X_valid_d, y_train_d, y_valid_d = train_test_split(X_diabetes, y_diabetes, test_size=.2, random_state=RANDOM_STATE)
tree_regressors = {
    "Decision tree baseline": DecisionTreeRegressor(max_depth=4, random_state=RANDOM_STATE),
    "Bagging": BaggingRegressor(estimator=DecisionTreeRegressor(random_state=RANDOM_STATE), n_estimators=100, random_state=RANDOM_STATE, n_jobs=1),
    "Random forest": RandomForestRegressor(n_estimators=200, min_samples_leaf=5, random_state=RANDOM_STATE, n_jobs=1),
    "Gradient boosting": GradientBoostingRegressor(random_state=RANDOM_STATE),
    "BART": BART(num_trees=50, num_particles=5, max_stages=100, ndraw=20, burnin=20, random_state=RANDOM_STATE, n_jobs=1),
}
regression_rows = []
for name, model in tree_regressors.items():
    model.fit(X_train_d, y_train_d)
    pred = np.asarray(model.predict(X_valid_d)).reshape(-1)
    regression_rows.append({
        "textbook_model": name,
        "validation_rmse": np.sqrt(mean_squared_error(y_valid_d, pred)),
        "validation_r2": r2_score(y_valid_d, pred),
    })
tree_regression_results = pd.DataFrame(regression_rows).sort_values("validation_rmse")
display(tree_regression_results)
best_tree_regression = tree_regression_results.iloc[0]["textbook_model"]
display(pd.DataFrame({
    "part": ["Applied Question 12"],
    "best_validation_model": [best_tree_regression],
    "interpretation": ["The best held-out RMSE identifies the strongest tree-family regression model for this chosen data set; BART is included as the Bayesian additive tree comparison required by the textbook exercise."],
}))

,textbook_model,validation_rmse,validation_r2
4,BART,51.833426,0.492898
2,Random forest,53.528279,0.459193
3,Gradient boosting,53.837131,0.452934
1,Bagging,54.505626,0.439264
0,Decision tree baseline,59.740817,0.326375


,part,best_validation_model,interpretation
0,Applied Question 12,BART,The best held-out RMSE identifies the stronges...


The textbook applied question is intentionally separate from the Kaggle classification requirement.  Bagging and random forests reduce variance by averaging many trees, boosting builds trees sequentially, and BART averages over posterior tree structures rather than fitting one fixed tree ensemble.  The RMSE and R-squared table provides the required comparison on a common validation split.

## Kaggle Obesity Tree Submissions

The Kaggle portion requires four tree-based classification submissions on the Obesity data.  This section remains separate from the BART regression comparison because the competition target is multiclass classification.  A shared preprocessing object and stratified validation split keep the four models on the same observations.

In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import BaggingClassifier, GradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier

obesity = pd.read_csv(KAGGLE / "playground-series-s4e2" / "train.csv")
X = obesity.drop(columns=["NObeyesdad"])
y = obesity["NObeyesdad"]
display(y.value_counts(normalize=True).rename("class_share").to_frame())
cat = X.select_dtypes(include="object").columns.tolist()
num = [c for c in X.columns if c not in cat + ["id"]]
pre = ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), cat)], remainder="passthrough")
X_train, X_valid, y_train, y_valid = train_test_split(X.drop(columns=["id"]), y, test_size=.2, stratify=y, random_state=RANDOM_STATE)
base_tree = DecisionTreeClassifier(max_depth=8, random_state=RANDOM_STATE)
display(pd.DataFrame({
    "setup_check": ["training_rows", "validation_rows", "class_count", "categorical_predictors", "numeric_predictors"],
    "value": [len(X_train), len(X_valid), y.nunique(), len(cat), len(num)],
}))

,class_share
NObeyesdad,
Obesity_Type_III,0.194913
Obesity_Type_II,0.156470
Normal_Weight,0.148473
Obesity_Type_I,0.140187
Insufficient_Weight,0.121544
Overweight_Level_II,0.121495
Overweight_Level_I,0.116919


,setup_check,value
0,training_rows,16606
1,validation_rows,4152
2,class_count,7
3,categorical_predictors,8
4,numeric_predictors,8


### Kaggle Model 1: Decision Tree

The first Kaggle tree model is a single decision tree.  It gives the clearest baseline because the prediction is driven by one sequence of splits.

In [4]:
tree_model = Pipeline([
    ("pre", pre),
    ("model", DecisionTreeClassifier(max_depth=8, random_state=RANDOM_STATE)),
])
tree_model.fit(X_train, y_train)
tree_train_pred = tree_model.predict(X_train)
tree_pred = tree_model.predict(X_valid)
tree_train_accuracy = accuracy_score(y_train, tree_train_pred)
tree_accuracy = accuracy_score(y_valid, tree_pred)
tree_balanced = balanced_accuracy_score(y_valid, tree_pred)
display(pd.DataFrame({
    "model": ["Decision tree"],
    "train_accuracy": [tree_train_accuracy],
    "validation_accuracy": [tree_accuracy],
    "balanced_accuracy": [tree_balanced],
    "generalization_gap": [tree_train_accuracy - tree_accuracy],
    "submission_file": ["A6_decision_tree_playground_series_s4e2.csv"],
}))

,model,train_accuracy,validation_accuracy,balanced_accuracy,generalization_gap,submission_file
0,Decision tree,0.893171,0.869461,0.856238,0.023711,A6_decision_tree_playground_series_s4e2.csv


The single tree is useful because it is interpretable, but its role is mainly as a baseline.  A large train-validation gap would signal that one tree is adapting too closely to the training split.  Even when the gap is moderate, a single tree is more sensitive to small data changes than the ensemble models that follow.

### Kaggle Model 2: Bagging

The second Kaggle tree model is bagging.  It keeps the decision-tree base learner but averages many bootstrap trees to reduce variance.

In [5]:
bagging_model = Pipeline([
    ("pre", pre),
    ("model", BaggingClassifier(estimator=base_tree, n_estimators=80, random_state=RANDOM_STATE, n_jobs=1)),
])
bagging_model.fit(X_train, y_train)
bagging_train_pred = bagging_model.predict(X_train)
bagging_pred = bagging_model.predict(X_valid)
bagging_train_accuracy = accuracy_score(y_train, bagging_train_pred)
bagging_accuracy = accuracy_score(y_valid, bagging_pred)
bagging_balanced = balanced_accuracy_score(y_valid, bagging_pred)
display(pd.DataFrame({
    "model": ["Bagging"],
    "train_accuracy": [bagging_train_accuracy],
    "validation_accuracy": [bagging_accuracy],
    "balanced_accuracy": [bagging_balanced],
    "generalization_gap": [bagging_train_accuracy - bagging_accuracy],
    "submission_file": ["A6_bagging_playground_series_s4e2.csv"],
}))

,model,train_accuracy,validation_accuracy,balanced_accuracy,generalization_gap,submission_file
0,Bagging,0.902565,0.888006,0.875682,0.01456,A6_bagging_playground_series_s4e2.csv


Bagging improves the research story because it directly tests whether the instability of a single tree is hurting performance.  If validation accuracy improves while the gap stays controlled, the improvement is evidence that averaging many trees is reducing variance rather than only memorizing the training data (Breiman, 1996).

### Kaggle Model 3: Random Forest

The third Kaggle tree model is a random forest.  It extends bagging by adding feature randomness, which lowers correlation among the trees being averaged.

In [6]:
rf_model = Pipeline([
    ("pre", pre),
    ("model", RandomForestClassifier(n_estimators=150, max_depth=12, random_state=RANDOM_STATE, n_jobs=1)),
])
rf_model.fit(X_train, y_train)
rf_train_pred = rf_model.predict(X_train)
rf_pred = rf_model.predict(X_valid)
rf_train_accuracy = accuracy_score(y_train, rf_train_pred)
rf_accuracy = accuracy_score(y_valid, rf_pred)
rf_balanced = balanced_accuracy_score(y_valid, rf_pred)
rf_feature_names = rf_model.named_steps["pre"].get_feature_names_out()
rf_importance = pd.DataFrame({
    "feature": rf_feature_names,
    "importance": rf_model.named_steps["model"].feature_importances_,
}).sort_values("importance", ascending=False).head(12)
display(pd.DataFrame({
    "model": ["Random forest"],
    "train_accuracy": [rf_train_accuracy],
    "validation_accuracy": [rf_accuracy],
    "balanced_accuracy": [rf_balanced],
    "generalization_gap": [rf_train_accuracy - rf_accuracy],
    "submission_file": ["A6_random_forest_playground_series_s4e2.csv"],
}))
display(rf_importance)

,model,train_accuracy,validation_accuracy,balanced_accuracy,generalization_gap,submission_file
0,Random forest,0.952668,0.890173,0.877267,0.062494,A6_random_forest_playground_series_s4e2.csv


,feature,importance
24,remainder__Weight,0.359729
25,remainder__FCVC,0.100077
22,remainder__Age,0.093350
23,remainder__Height,0.074393
0,cat__Gender_Female,0.045141
29,remainder__TUE,0.045004
27,remainder__CH2O,0.038828
1,cat__Gender_Male,0.035775
26,remainder__NCP,0.027150
28,remainder__FAF,0.025871


The random forest is the strongest variance-reduction comparison because it changes not only the sampled rows but also the predictors considered at each split.  The feature-importance table gives the model a substantive interpretation and shows which encoded health and measurement variables are driving the classification result (Breiman, 2001).

### Kaggle Model 4: Gradient Boosting

The fourth Kaggle tree model is gradient boosting.  It builds trees sequentially, so it is designed to correct errors left by earlier trees rather than average trees fit independently.

In [7]:
gb_model = Pipeline([
    ("pre", pre),
    ("model", GradientBoostingClassifier(random_state=RANDOM_STATE)),
])
gb_model.fit(X_train, y_train)
gb_train_pred = gb_model.predict(X_train)
gb_pred = gb_model.predict(X_valid)
gb_train_accuracy = accuracy_score(y_train, gb_train_pred)
gb_accuracy = accuracy_score(y_valid, gb_pred)
gb_balanced = balanced_accuracy_score(y_valid, gb_pred)
gb_feature_names = gb_model.named_steps["pre"].get_feature_names_out()
gb_importance = pd.DataFrame({
    "feature": gb_feature_names,
    "importance": gb_model.named_steps["model"].feature_importances_,
}).sort_values("importance", ascending=False).head(12)
display(pd.DataFrame({
    "model": ["Gradient boosting"],
    "train_accuracy": [gb_train_accuracy],
    "validation_accuracy": [gb_accuracy],
    "balanced_accuracy": [gb_balanced],
    "generalization_gap": [gb_train_accuracy - gb_accuracy],
    "submission_file": ["A6_gradient_boosting_playground_series_s4e2.csv"],
}))
display(gb_importance)

,model,train_accuracy,validation_accuracy,balanced_accuracy,generalization_gap,submission_file
0,Gradient boosting,0.921233,0.905347,0.894882,0.015886,A6_gradient_boosting_playground_series_s4e2.csv


,feature,importance
24,remainder__Weight,0.608735
25,remainder__FCVC,0.111034
23,remainder__Height,0.072169
1,cat__Gender_Male,0.049671
0,cat__Gender_Female,0.048556
22,remainder__Age,0.036326
27,remainder__CH2O,0.016228
16,cat__CALC_no,0.013646
26,remainder__NCP,0.011562
29,remainder__TUE,0.007041


Gradient boosting is the strongest recorded Kaggle submission in this assignment.  The interpretation is not simply that it has a higher score; the sequential fitting process is better matched to a multiclass problem where difficult boundary cases remain after the first trees are fit (Friedman, 2001).

In [8]:
kaggle_a6_comparison = pd.DataFrame([
    {"model": "Decision tree", "train_accuracy": tree_train_accuracy, "validation_accuracy": tree_accuracy, "balanced_accuracy": tree_balanced, "generalization_gap": tree_train_accuracy - tree_accuracy, "submission_file": "A6_decision_tree_playground_series_s4e2.csv"},
    {"model": "Bagging", "train_accuracy": bagging_train_accuracy, "validation_accuracy": bagging_accuracy, "balanced_accuracy": bagging_balanced, "generalization_gap": bagging_train_accuracy - bagging_accuracy, "submission_file": "A6_bagging_playground_series_s4e2.csv"},
    {"model": "Random forest", "train_accuracy": rf_train_accuracy, "validation_accuracy": rf_accuracy, "balanced_accuracy": rf_balanced, "generalization_gap": rf_train_accuracy - rf_accuracy, "submission_file": "A6_random_forest_playground_series_s4e2.csv"},
    {"model": "Gradient boosting", "train_accuracy": gb_train_accuracy, "validation_accuracy": gb_accuracy, "balanced_accuracy": gb_balanced, "generalization_gap": gb_train_accuracy - gb_accuracy, "submission_file": "A6_gradient_boosting_playground_series_s4e2.csv"},
]).sort_values("validation_accuracy", ascending=False)
display(kaggle_a6_comparison)
best_tree_name = kaggle_a6_comparison.iloc[0]["model"]
best_tree_pred = {"Decision tree": tree_pred, "Bagging": bagging_pred, "Random forest": rf_pred, "Gradient boosting": gb_pred}[best_tree_name]
display(pd.DataFrame(confusion_matrix(y_valid, best_tree_pred, labels=tree_model.classes_), index=tree_model.classes_, columns=tree_model.classes_))

,model,train_accuracy,validation_accuracy,balanced_accuracy,generalization_gap,submission_file
3,Gradient boosting,0.921233,0.905347,0.894882,0.015886,A6_gradient_boosting_playground_series_s4e2.csv
2,Random forest,0.952668,0.890173,0.877267,0.062494,A6_random_forest_playground_series_s4e2.csv
1,Bagging,0.902565,0.888006,0.875682,0.014560,A6_bagging_playground_series_s4e2.csv
0,Decision tree,0.893171,0.869461,0.856238,0.023711,A6_decision_tree_playground_series_s4e2.csv


,Insufficient_Weight,Normal_Weight,Obesity_Type_I,Obesity_Type_II,Obesity_Type_III,Overweight_Level_I,Overweight_Level_II
Insufficient_Weight,482,22,0,0,0,1,0
Normal_Weight,32,547,1,0,0,29,8
Obesity_Type_I,1,3,523,14,3,7,31
Obesity_Type_II,1,0,20,628,0,0,1
Obesity_Type_III,0,0,2,1,806,0,0
Overweight_Level_I,1,48,9,0,0,365,62
Overweight_Level_II,0,7,41,3,0,45,408


The comparison table places the four Kaggle submissions on the same validation split.  The single tree anchors interpretability, bagging and random forests test variance reduction, and gradient boosting tests sequential error correction.  The confusion matrix for the best validation model makes the remaining class confusion visible instead of leaving the conclusion at the accuracy level.

In [9]:
import re

status_path = SUBMISSIONS / "kaggle_submission_status_playground-series-s4e2.txt"
for encoding in ("utf-16", "utf-8"):
    try:
        text = status_path.read_text(encoding=encoding)
        break
    except UnicodeError:
        continue

records = []
for line in text.splitlines():
    parts = re.split(r"\s{2,}", line.strip())
    if len(parts) >= 7 and parts[0].isdigit() and "A6_" in parts[1]:
        records.append({
            "ref": parts[0],
            "fileName": parts[1],
            "date": parts[2],
            "description": parts[3],
            "status": parts[4],
            "publicScore": parts[5],
            "privateScore": parts[6],
        })

display(pd.DataFrame(records))
display(pd.DataFrame({
    "submission_file": ['A6_decision_tree_playground_series_s4e2.csv', 'A6_bagging_playground_series_s4e2.csv', 'A6_random_forest_playground_series_s4e2.csv', 'A6_gradient_boosting_playground_series_s4e2.csv'],
    "exists_locally": [(SUBMISSIONS / file).exists() for file in ['A6_decision_tree_playground_series_s4e2.csv', 'A6_bagging_playground_series_s4e2.csv', 'A6_random_forest_playground_series_s4e2.csv', 'A6_gradient_boosting_playground_series_s4e2.csv']],
}))
display(pd.DataFrame({
    "evidence": ["Public GitHub repository", "Notebook path in repository"],
    "value": [
        "https://github.com/maldo81/dds-8555-predictive-analysis",
        "Week 6/Assignment 6/MaldonadoJDDS8555-6.ipynb",
    ],
}))

,ref,fileName,date,description,status,publicScore,privateScore
0,52966810,A6_gradient_boosting_playground_series_s4e2.csv,2026-05-23 21:27:41.547000,DDS-8555 A6 boosted tree model,SubmissionStatus.COMPLETE,0.90462,0.90245
1,52966809,A6_random_forest_playground_series_s4e2.csv,2026-05-23 21:27:39.150000,DDS-8555 A6 random forest model,SubmissionStatus.COMPLETE,0.88945,0.89315
2,52966808,A6_bagging_playground_series_s4e2.csv,2026-05-23 21:27:36.670000,DDS-8555 A6 bagged tree model,SubmissionStatus.COMPLETE,0.89450,0.89342
3,52966806,A6_decision_tree_playground_series_s4e2.csv,2026-05-23 21:27:34.260000,DDS-8555 A6 decision tree model,SubmissionStatus.COMPLETE,0.87680,0.87030


,submission_file,exists_locally
0,A6_decision_tree_playground_series_s4e2.csv,True
1,A6_bagging_playground_series_s4e2.csv,True
2,A6_random_forest_playground_series_s4e2.csv,True
3,A6_gradient_boosting_playground_series_s4e2.csv,True


,evidence,value
0,Public GitHub repository,https://github.com/maldo81/dds-8555-predictive...
1,Notebook path in repository,Week 6/Assignment 6/MaldonadoJDDS8555-6.ipynb


## Interpretation

The tree family shows the bias-variance trade-off in a practical way.  A single decision tree is interpretable but unstable.  Bagging reduces variance by averaging many trees, random forests add feature randomness to reduce tree correlation, and boosting builds a sequence of trees that focus on difficult cases (Breiman, 1996, 2001; Friedman, 2001).  The BART comparison extends the textbook applied question by adding a Bayesian additive tree model on the diabetes regression data.  The Kaggle validation table includes the train-validation gap so overfitting is visible rather than assumed.  The confusion matrix and feature-importance table give the best validation model a direct interpretation.  The Kaggle evidence shows that all four required submissions completed, with boosted trees producing the best public score among this group.  As in Assignment 5, these are competition obesity labels rather than clinical diagnoses (NCD Risk Factor Collaboration, 2016).

## References

Breiman, L. (1996).  Bagging predictors. *Machine Learning, 24*, 123-140. https://doi.org/10.1007/BF00058655

Breiman, L. (2001).  Random forests. *Machine Learning, 45*, 5-32. https://doi.org/10.1023/A:1010933404324

Chipman, H.  A., George, E.  I., & McCulloch, R.  E. (2010).  BART: Bayesian additive regression trees. *The Annals of Applied Statistics, 4*(1), 266-298. https://doi.org/10.1214/09-AOAS285

Friedman, J.  H. (2001).  Greedy function approximation: A gradient boosting machine. *The Annals of Statistics, 29*(5), 1189-1232. https://doi.org/10.1214/aos/1013203451

NCD Risk Factor Collaboration. (2016).  Trends in adult body-mass index in 200 countries from 1975 to 2014: A pooled analysis of 1698 population-based measurement studies with 19.2 million participants. *The Lancet, 387*(10026), 1377-1396. https://doi.org/10.1016/S0140-6736(16)30054-X